In [2]:
import matplotlib
matplotlib.use('Agg') # Set non-interactive backend before importing pyplot
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.animation as animation # Import animation module
from matplotlib.backends.backend_agg import FigureCanvasAgg # Import FigureCanvasAgg
from matplotlib.figure import Figure # Import Figure directly
import time
import numpy as np
import random
import os
import subprocess # Import subprocess for direct ffmpeg call

# Set up matplotlib to use ffmpeg for video writing
# Ensure ffmpeg is installed and its path is correctly set. In Colab, it's usually /usr/bin/ffmpeg.
plt.rcParams['animation.ffmpeg_path'] = '/usr/bin/ffmpeg'

# --- Car Class Definition ---
class Car:
    CAR_LENGTH = 0.8 # Class attribute

    def __init__(self, car_id, lane_idx, offset_from_stop_line):
        self.car_id = car_id
        self.lane_idx = lane_idx
        self.offset_from_stop_line = offset_from_stop_line
        # Add a property to store the initial position when cleared for animation purposes
        self.cleared_from_offset = None
        self.cleared_from_lane_idx = None

    def get_xy_coords(self, lane_data):
        stop_line_coords = lane_data['stop_line_coords']
        direction_vector = lane_data['direction_vector']

        # Calculate global (x, y) coordinates
        x = stop_line_coords[0] + self.offset_from_stop_line * direction_vector[0]
        y = stop_line_coords[1] + self.offset_from_stop_line * direction_vector[1]
        return (x, y)


# --- SimpleTrafficEnv Class Definition ---
class SimpleTrafficEnv:
    """
    A simplified, non-visual environment for a single 4-way intersection.
    The state is a set of queue lengths.
    """

    CAR_LENGTH = 0.8
    CAR_SPACING = 0.2

    # Class-level dictionary for lane data
    LANE_DATA = {
        0: {'stop_line_coords': (0, -2), 'direction_vector': (0, -1)}, # North lane
        1: {'stop_line_coords': (0, 2), 'direction_vector': (0, 1)},  # South lane
        2: {'stop_line_coords': (-2, 0), 'direction_vector': (-1, 0)}, # East lane
        3: {'stop_line_coords': (2, 0), 'direction_vector': (1, 0)}   # West lane
    }

    def __init__(self, num_lanes=4):
        self.num_lanes = num_lanes
        self.action_space_size = 2 # Action 0: N-S Green; Action 1: E-W Green
        self.queue_bins = [5, 10, 15] # Thresholds for discretizing queue lengths

        # Replace self.current_queues with self.cars_in_lanes
        self.cars_in_lanes = [[] for _ in range(self.num_lanes)]
        self.car_id_counter = 0 # Initialize car ID counter

    def reset(self):
        """Initializes a new traffic scenario (episode)."""
        self.cars_in_lanes = [[] for _ in range(self.num_lanes)] # Reset lanes
        self.car_id_counter = 0 # Reset car ID counter

        for lane_idx in range(self.num_lanes):
            initial_cars_count = random.randint(5, 20) # Random initial number of cars
            for i in range(initial_cars_count):
                # Calculate offset for each car
                offset_from_stop_line = i * (self.CAR_LENGTH + self.CAR_SPACING)
                # Create Car object and add to lane
                new_car = Car(self.car_id_counter, lane_idx, offset_from_stop_line)
                self.cars_in_lanes[lane_idx].append(new_car)
                self.car_id_counter += 1

        # For compatibility with visualization code that still expects current_queues
        self.current_queues = np.array([len(cars) for cars in self.cars_in_lanes])

        return self._get_discrete_state()

    def _get_discrete_state(self):
        """Converts continuous queue lengths into a discrete state (tuple of bins)."""
        discrete_state = []
        # Now, queue length is the number of cars in the lane
        current_queue_lengths = [len(cars) for cars in self.cars_in_lanes]
        for q_len in current_queue_lengths:
            bin_index = 0
            for i, threshold in enumerate(self.queue_bins):
                if q_len >= threshold:
                    bin_index = i + 1
            discrete_state.append(bin_index)
        return tuple(discrete_state)

    def step(self, action):
        """Executes a traffic light phase change and advances the simulation."""

        cleared_cars_in_step = [] # List to store cars that clear the intersection

        # --- A. Apply Action (Clear Traffic) ---
        if action == 0: # N-S Green (Clears lanes 0 and 1)
            clearance = random.randint(3, 7)
            # Clear North lane
            num_to_clear_n = min(len(self.cars_in_lanes[0]), clearance)
            for car in self.cars_in_lanes[0][:num_to_clear_n]:
                car.cleared_from_offset = car.offset_from_stop_line
                car.cleared_from_lane_idx = 0
                cleared_cars_in_step.append(car)
            self.cars_in_lanes[0] = self.cars_in_lanes[0][num_to_clear_n:]

            # Clear South lane
            num_to_clear_s = min(len(self.cars_in_lanes[1]), clearance)
            for car in self.cars_in_lanes[1][:num_to_clear_s]:
                car.cleared_from_offset = car.offset_from_stop_line
                car.cleared_from_lane_idx = 1
                cleared_cars_in_step.append(car)
            self.cars_in_lanes[1] = self.cars_in_lanes[1][num_to_clear_s:]

        else: # E-W Green (Clears lanes 2 and 3)
            clearance = random.randint(3, 7)
            # Clear East lane
            num_to_clear_e = min(len(self.cars_in_lanes[2]), clearance)
            for car in self.cars_in_lanes[2][:num_to_clear_e]:
                car.cleared_from_offset = car.offset_from_stop_line
                car.cleared_from_lane_idx = 2
                cleared_cars_in_step.append(car)
            self.cars_in_lanes[2] = self.cars_in_lanes[2][num_to_clear_e:]

            # Clear West lane
            num_to_clear_w = min(len(self.cars_in_lanes[3]), clearance)
            for car in self.cars_in_lanes[3][:num_to_clear_w]:
                car.cleared_from_offset = car.offset_from_stop_line
                car.cleared_from_lane_idx = 3
                cleared_cars_in_step.append(car)
            self.cars_in_lanes[3] = self.cars_in_lanes[3][num_to_clear_w:]

        # Update offsets of remaining cars
        for lane_idx in range(self.num_lanes):
            for i, car in enumerate(self.cars_in_lanes[lane_idx]):
                car.offset_from_stop_line = i * (self.CAR_LENGTH + self.CAR_SPACING)

        # --- B. Vehicle Arrivals (New Congestion) ---
        # Cars arrive on all lanes regardless of light color
        for i in range(self.num_lanes):
            num_arrivals = random.randint(0, 3)
            for _ in range(num_arrivals):
                # New car arrives at the back of the queue
                current_queue_len = len(self.cars_in_lanes[i])
                offset_from_stop_line = current_queue_len * (self.CAR_LENGTH + self.CAR_SPACING)
                new_car = Car(self.car_id_counter, i, offset_from_stop_line)
                self.cars_in_lanes[i].append(new_car)
                self.car_id_counter += 1

        # --- C. Calculate Reward ---
        # Reward is the NEGATIVE sum of SQUARED current queue lengths (goal is to maximize this, i.e., minimize queue and balance them)
        current_queue_lengths_array = np.array([len(cars) for cars in self.cars_in_lanes])
        reward = -np.sum(np.square(current_queue_lengths_array))

        new_discrete_state = self._get_discrete_state()
        done = False # Continuous process

        # For visualization purposes, we need to return the 'current_queues' equivalent
        # which is simply the length of car lists now.
        self.current_queues = current_queue_lengths_array

        info = {
            'cleared_cars': cleared_cars_in_step
        }

        return new_discrete_state, reward, done, info

# Helper functions for car tracking changes
def prev_cars_snapshot_to_current_cars(prev_snapshot, current_cars_in_lanes, current_car_map_for_frame=None):
    # This helper simulates the behavior of ArtistAnimation in tracking cars that existed before
    # and still exist after the step, potentially with updated positions.
    # It returns a list of (prev_car, current_car) tuples.

    if current_car_map_for_frame is None:
        # Build current car map if not provided (optimization to avoid re-building in loops)
        current_car_map_for_frame = {}
        for lane in current_cars_in_lanes:
            for car in lane:
                current_car_map_for_frame[car.car_id] = car

    cars_remaining_and_moving = []
    for prev_lane in prev_snapshot:
        for prev_car in prev_lane:
            if prev_car.car_id in current_car_map_for_frame:
                current_car = current_car_map_for_frame[prev_car.car_id]
                cars_remaining_and_moving.append((prev_car, current_car))
    return cars_remaining_and_moving

def get_newly_arrived_cars(prev_snapshot, current_cars_in_lanes):
    # Returns cars that are in current_cars_in_lanes but not in prev_snapshot
    prev_car_ids = set()
    for lane in prev_snapshot:
        for car in lane:
            prev_car_ids.add(car.car_id)

    newly_arrived = []
    for lane in current_cars_in_lanes:
        for car in lane:
            if car.car_id not in prev_car_ids:
                newly_arrived.append(car)
    return newly_arrived

def _draw_single_frame(ax, env, action_desc,
                         current_cars_in_lanes, cleared_cars_in_step,
                         cars_remaining_and_moving, cars_newly_arrived,
                         animation_progress, step_num, total_steps,
                         Car_Class=Car):

    ax.clear() # Clear previous frame
    ax.set_xlim(-10, 10)
    ax.set_ylim(-10, 10)
    ax.set_aspect('equal', adjustable='box')
    ax.axis('off') # Hide axes

    # Draw roads
    ax.add_patch(patches.Rectangle((-2, -10), 4, 20, color='gray'))
    ax.add_patch(patches.Rectangle((-10, -2), 20, 4, color='gray'))

    # Draw intersection
    ax.add_patch(patches.Rectangle((-2, -2), 4, 4, color='darkgray'))

    # Draw traffic lights
    light_color_ns = 'green' if action_desc == 'N-S Green' else 'red'
    light_color_ew = 'green' if action_desc == 'E-W Green' else 'red'
    ax.add_patch(patches.Circle((0, 3), 0.5, color=light_color_ns)) # North light
    ax.add_patch(patches.Circle((0, -3), 0.5, color=light_color_ns)) # South light
    ax.add_patch(patches.Circle((3, 0), 0.5, color=light_color_ew)) # East light
    ax.add_patch(patches.Circle((-3, 0), 0.5, color=light_color_ew)) # West light

    # 1. Animate cars that cleared the intersection
    for car in cleared_cars_in_step:
        lane_data = env.LANE_DATA[car.cleared_from_lane_idx]
        start_offset = car.cleared_from_offset

        # Define end_offset to move cleared cars off-screen. Need to be consistent with direction.
        if car.cleared_from_lane_idx == 0: # North lane (cars move south, i.e., direction_vector (0, -1))
            end_offset_cleared = -10  # Moves in negative Y direction
        elif car.cleared_from_lane_idx == 1: # South lane (cars move north, i.e., direction_vector (0, 1))
            end_offset_cleared = 10   # Moves in positive Y direction
        elif car.cleared_from_lane_idx == 2: # East lane (cars move west, i.e., direction_vector (-1, 0))
            end_offset_cleared = -10  # Moves in negative X direction
        elif car.cleared_from_lane_idx == 3: # West lane (cars move east, i.e., direction_vector (1, 0))
            end_offset_cleared = 10   # Moves in positive X direction

        current_offset = start_offset + (end_offset_cleared - start_offset) * animation_progress
        temp_car = Car_Class(car.car_id, car.cleared_from_lane_idx, current_offset)
        x, y = temp_car.get_xy_coords(lane_data)
        # Only draw if the car is still somewhat visible (not completely off screen)
        if -15 < x < 15 and -15 < y < 15: # Simple bounds check
            ax.add_patch(patches.Rectangle((x - Car_Class.CAR_LENGTH/2, y - Car_Class.CAR_LENGTH/2), Car_Class.CAR_LENGTH, Car_Class.CAR_LENGTH, color='purple', alpha=0.8))

    # 2. Animate cars that remain in queues and shift forward
    for prev_car, current_car in cars_remaining_and_moving:
        lane_data = env.LANE_DATA[current_car.lane_idx]
        start_offset = prev_car.offset_from_stop_line
        end_offset = current_car.offset_from_stop_line

        current_offset = start_offset + (end_offset - start_offset) * animation_progress
        temp_car = Car_Class(current_car.car_id, current_car.lane_idx, current_offset)
        x, y = temp_car.get_xy_coords(lane_data)
        ax.add_patch(patches.Rectangle((x - Car_Class.CAR_LENGTH/2, y - Car_Class.CAR_LENGTH/2), Car_Class.CAR_LENGTH, Car_Class.CAR_LENGTH, color='blue'))

    # 3. Draw newly arrived cars - only appear in the final frame of the animation step
    if animation_progress == 1.0: # Only draw new cars at the end of the animation step (final frame)
        for car in cars_newly_arrived:
            lane_data = env.LANE_DATA[car.lane_idx]
            x, y = car.get_xy_coords(lane_data)
            ax.add_patch(patches.Rectangle((x - Car_Class.CAR_LENGTH/2, y - Car_Class.CAR_LENGTH/2), Car_Class.CAR_LENGTH, Car_Class.CAR_LENGTH, color='green', alpha=0.8))

    ax.set_title(f'Animated Traffic Simulation: Step {step_num}/{total_steps}')

def test_policy_visual_animated(env, q_table, steps=10, animation_frames=5, filename='traffic_animation.mp4'):
    fig, ax = plt.subplots(figsize=(8, 8))
    temp_frame_dir = 'temp_animation_frames'
    os.makedirs(temp_frame_dir, exist_ok=True)

    d_state = env.reset()
    prev_cars_snapshot = [list(lane) for lane in env.cars_in_lanes] # Deep copy of initial state
    frame_count = 0

    print(f"Generating frames for {steps} simulation steps...")
    for step_num in range(1, steps + 1):
        action = np.argmax(q_table[d_state]) # Exploit learned policy
        action_desc = "N-S Green" if action == 0 else "E-W Green"

        new_d_state, reward, done, info = env.step(action)
        cleared_cars_in_step = info.get('cleared_cars', [])

        current_car_map = {car.car_id: car for lane in env.cars_in_lanes for car in lane}
        cars_remaining_and_moving_for_step = prev_cars_snapshot_to_current_cars(prev_cars_snapshot, env.cars_in_lanes, current_car_map)
        cars_newly_arrived_for_step = get_newly_arrived_cars(prev_cars_snapshot, env.cars_in_lanes)

        for frame_idx in range(animation_frames + 1): # +1 to include final state for this step
            animation_progress = frame_idx / animation_frames
            _draw_single_frame(ax, env, action_desc,
                               env.cars_in_lanes, cleared_cars_in_step,
                               cars_remaining_and_moving_for_step, cars_newly_arrived_for_step,
                               animation_progress, step_num, steps)
            fig.savefig(os.path.join(temp_frame_dir, f'frame_{frame_count:04d}.png'), dpi=150)
            frame_count += 1

        d_state = new_d_state
        prev_cars_snapshot = [list(lane) for lane in env.cars_in_lanes] # Update snapshot for next step

        if done:
            break

    print(f"Total images saved: {frame_count}")
    print(f"Stitching frames into {filename} with ffmpeg...")
    try:
        # FFMpeg command to stitch frames into a video
        ffmpeg_command = [
            'ffmpeg',
            '-y', # Overwrite output file if it exists
            '-framerate', '10', # 10 frames per second for the output video (0.5x speed of 20)
            '-i', os.path.join(temp_frame_dir, 'frame_%04d.png'),
            '-c:v', 'libx264',
            '-pix_fmt', 'yuv420p',
            '-vf', 'pad=ceil(iw/2)*2:ceil(ih/2)*2', # Ensures even dimensions for some codecs
            filename
        ]
        subprocess.run(ffmpeg_command, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        print(f"Animation saved successfully to {filename}.")
    except subprocess.CalledProcessError as e:
        print(f"Error saving animation with ffmpeg: {e}")
        print(f"FFmpeg stdout: {e.stdout.decode()}")
        print(f"FFmpeg stderr: {e.stderr.decode()}")
        print("Please ensure ffmpeg is correctly installed and accessible in your environment.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
    finally:
        # Clean up temporary frame files
        for f in os.listdir(temp_frame_dir):
            os.remove(os.path.join(temp_frame_dir, f))
        os.rmdir(temp_frame_dir)
        plt.close(fig)

In [ ]:
import sys
import subprocess

def check_and_install_ffmpeg():
    try:
        subprocess.run(["ffmpeg", "-version"], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        print("ffmpeg is already installed.")
    except (subprocess.CalledProcessError, FileNotFoundError):
        print("ffmpeg not found. Installing...")
        try:
            # Try installing with apt-get for Debian/Ubuntu based systems (common in Colab)
            subprocess.run(["apt-get", "update"], check=True)
            subprocess.run(["apt-get", "install", "ffmpeg", "-y"], check=True)
            print("ffmpeg installed successfully using apt-get.")
        except (subprocess.CalledProcessError, FileNotFoundError):
            print("Could not install ffmpeg using apt-get. Please install it manually or check your environment.")

check_and_install_ffmpeg()

ffmpeg is already installed.


In [3]:
import numpy as np
import random
import matplotlib.pyplot as plt

# The Car and SimpleTrafficEnv classes are defined in cell `6c94f46f`.
# Ensure cell `6c94f46f` is run before this cell.

# --- Hyperparameters ---
LEARNING_RATE = 0.1     # alpha
DISCOUNT_FACTOR = 0.95  # gamma
EPISODES = 500          # Number of training rounds
MAX_STEPS = 50          # Max simulation steps per episode
EPSILON_START = 1.0     # Starting exploration rate
EPSILON_END = 0.01
EPSILON_DECAY = 0.995

# --- Initialize Environment and Q-Table ---
env = SimpleTrafficEnv() # Initialize the environment (defined in 6c94f46f)

# State space size: 4 lanes, 4 bins per lane (0 to 3) -> 4^4 = 256 states
# Q-Table dimensions: (Bin_L1, Bin_L2, Bin_L3, Bin_L4, Action)
# Initialize Q_TABLE with zeros for a fresh training
Q_TABLE = np.zeros(tuple([len(env.queue_bins) + 1] * env.num_lanes + [env.action_space_size]))

epsilon = EPSILON_START
total_rewards = []

# --- Training Loop ---
print("Starting Q-Learning Training (with squared queue reward function)...")
for episode in range(EPISODES):
    d_state = env.reset() # Get initial discrete state
    episode_reward = 0

    for step in range(MAX_STEPS):
        # --- Epsilon-Greedy Strategy ---
        if random.random() < epsilon:
            action = random.randint(0, env.action_space_size - 1) # Explore
        else:
            action = np.argmax(Q_TABLE[d_state]) # Exploit

        # Take action and observe result
        new_d_state, reward, done, _ = env.step(action) # env.step now uses squared reward

        # --- Q-Table Update (Bellman Equation) ---
        old_value = Q_TABLE[d_state + (action,)]
        next_max = np.max(Q_TABLE[new_d_state])

        # Q(s, a) = (1 - alpha) * Q(s, a) + alpha * (R + gamma * max Q(s', a'))
        new_q_value = (1 - LEARNING_RATE) * old_value + LEARNING_RATE * (reward + DISCOUNT_FACTOR * next_max)#bellman equation        Q_TABLE[d_state + (action,)] = new_q_value

        d_state = new_d_state
        episode_reward += reward

        if done:
            break

    # Decay epsilon
    epsilon = max(EPSILON_END, epsilon * EPSILON_DECAY)
    total_rewards.append(episode_reward)

    if (episode + 1) % 50 == 0:
        print(f"Episode: {episode + 1}/{EPISODES} | Epsilon: {epsilon:.4f} | Avg Reward (last 50): {np.mean(total_rewards[-50:]):.2f}")

print("Training finished.")

# --- Save the trained Q-Table ---
np.save('q_table_squared_reward.npy', Q_TABLE)
print("Trained Q-Table saved to q_table_squared_reward.npy")

# --- Visualization of Learning (Plotting Reward) ---
plt.figure(figsize=(10, 6))
plt.plot(total_rewards)
plt.xlabel("Episode")
plt.ylabel("Cumulative Reward (Negative Sum of Squared Queue Lengths)")
plt.title("Q-Learning Agent Performance: Optimizing Traffic Flow (Squared Reward)")
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

Starting Q-Learning Training (with squared queue reward function)...
Episode: 50/500 | Epsilon: 0.7783 | Avg Reward (last 50): -12509.56
Episode: 100/500 | Epsilon: 0.6058 | Avg Reward (last 50): -32767.80
Episode: 150/500 | Epsilon: 0.4715 | Avg Reward (last 50): -45030.52
Episode: 200/500 | Epsilon: 0.3670 | Avg Reward (last 50): -86187.00
Episode: 250/500 | Epsilon: 0.2856 | Avg Reward (last 50): -123430.10
Episode: 300/500 | Epsilon: 0.2223 | Avg Reward (last 50): -159914.58
Episode: 350/500 | Epsilon: 0.1730 | Avg Reward (last 50): -178234.42
Episode: 400/500 | Epsilon: 0.1347 | Avg Reward (last 50): -210702.52
Episode: 450/500 | Epsilon: 0.1048 | Avg Reward (last 50): -213904.44
Episode: 500/500 | Epsilon: 0.0816 | Avg Reward (last 50): -244778.28
Training finished.
Trained Q-Table saved to q_table_squared_reward.npy


In [4]:
test_policy_visual_animated(env, Q_TABLE, steps=10, animation_frames=5, filename='optimized_traffic_simulation_squared_reward.mp4')

from google.colab import files

files.download('optimized_traffic_simulation_squared_reward.mp4')

Generating frames for 10 simulation steps...
Total images saved: 60
Stitching frames into optimized_traffic_simulation_squared_reward.mp4 with ffmpeg...
Animation saved successfully to optimized_traffic_simulation_squared_reward.mp4.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>